# RAG Playground

Interactive playground to test RAG retrieval and LLM integration.

In [1]:
import sys
from pathlib import Path

# Setup paths
backend_dir = Path.cwd().parent
sys.path.insert(0, str(backend_dir))

from app.services.rag_service import get_rag_service
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()
print("✓ Setup complete")

E:\Final Year Project\data_exploration_agent\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Setup complete


## 1. Initialize RAG Knowledge Base

In [2]:
from app.scripts.initialize_rag_kb import initialize_knowledge_bases

# Initialize all knowledge bases
initialize_knowledge_bases()

2026-01-09 19:13:28,312 - app.scripts.initialize_rag_kb - INFO - Starting RAG knowledge base initialization from markdown files...
2026-01-09 19:13:28,663 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-01-09 19:13:32,709 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-09 19:13:32,803 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-09 19:13:32,845 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-09 19:13:32,885 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for mo

## 2. Get RAG Service

In [4]:
rag = get_rag_service()
print("✓ RAG service loaded")

✓ RAG service loaded


## 3. Test Tool Selection Retrieval

In [6]:
# Query for tool guidance
query = "I want to know who painted the Mona Lisa"
step_goal = "Find the artist of a specific artwork"

results = rag.retrieve_tool_guidance(
    query=query,
    current_step_goal=step_goal,
    n_results=2
)

# Better way to inspect the results
print(f"Query: {query}\n")
for i, result in enumerate(results, 1):
    print(f"[{i}] Tool: {result['tool_name']} (score: {result['relevance_score']:.3f})")
    print(f"    Type: {result['metadata'].get('type')}")
    print(f"    Category: {result['metadata'].get('category')}")
    print(f"    Description: {result['metadata'].get('description')}")
    print(f"\n    Content preview:")
    print(f"    {result['content'][:200]}...\n")

2026-01-09 19:15:01,451 - app.services.rag_service - INFO - Retrieved 2 tool guidance entries


Query: I want to know who painted the Mona Lisa

[1] Tool: data_exploration_tool (score: 0.050)
    Type: tool_example
    Category: None
    Description: None

    Content preview:
    
Tool: data_exploration_tool
Scenario: Metadata Retrieval
User Query: Who painted 'Judith'?
Reasoning: 1. Artist information is stored in the database.
2. Requires querying the `artworks` table for th...

[2] Tool: data_exploration_tool (score: -0.063)
    Type: tool_example
    Category: None
    Description: None

    Content preview:
    
Tool: data_exploration_tool
Scenario: Finding Image Path
User Query: Analyze the colors in the 'Mona Lisa'.
Reasoning: 1. I need the file path of the image to analyze it.
2. The path is stored in the...



## 4. Test Explanation Pattern Retrieval

In [ ]:
# Query for explanation patterns
patterns = rag.retrieve_explanation_patterns(
    layer="planning",
    query="How do I analyze an image?",
    context={"tool_name": "image_analysis"},
    n_results=2
)

print("Explanation Patterns:\n")
for i, pattern in enumerate(patterns, 1):
    print(f"[{i}] {pattern['pattern'].get('pattern_name', 'Unknown')}")
    print(f"    Template: {pattern['pattern'].get('template', '')[:100]}...\n")

## 5. Test Error Explanation Retrieval

In [ ]:
# Query for error explanation
error_result = rag.retrieve_error_explanation(
    error_message="Table 'artworks' not found",
    tool_name="data_exploration_tool"
)

if error_result:
    print(f"Error Type: {error_result['error_type']}")
    print(f"Explanation: {error_result['explanation']}")
    print(f"Actions: {error_result['suggested_actions']}")
else:
    print("No error explanation found")

## 6. Test Domain Knowledge Retrieval

In [ ]:
# Query for domain knowledge
domain_results = rag.retrieve_domain_knowledge(
    query="How should I handle art style queries?",
    n_results=2
)

print("Domain Knowledge:\n")
for i, result in enumerate(domain_results, 1):
    print(f"[{i}] {result['category']} - {result['topic']}")
    print(f"    Score: {result['relevance_score']:.3f}\n")

## 7. RAG-Enhanced LLM Query

Test how RAG context improves LLM responses.

In [ ]:
# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert data exploration assistant.

Use the following knowledge to answer the user's question:

{context}

Provide a clear, concise answer based on the context."""),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()
print("✓ LLM chain ready")

In [ ]:
# Test query with RAG context
user_question = "Which tool should I use to find who painted the Starry Night?"

# Retrieve relevant context
tool_guidance = rag.retrieve_tool_guidance(
    query=user_question,
    current_step_goal="Select appropriate tool",
    n_results=2
)

# Format context
context = "\n\n".join([
    f"Tool: {r['tool_name']}\n{r['content']}"
    for r in tool_guidance
])

# Get LLM response
response = chain.invoke({
    "context": context,
    "question": user_question
})

print(f"Question: {user_question}\n")
print(f"Answer:\n{response}")

## 8. Compare: With vs Without RAG

In [ ]:
test_question = "Should I use image_analysis or data_exploration_tool to find the colors in a painting?"

# Without RAG
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{question}")
])
simple_chain = simple_prompt | llm | StrOutputParser()
without_rag = simple_chain.invoke({"question": test_question})

# With RAG
tool_context = rag.retrieve_tool_guidance(
    query=test_question,
    current_step_goal="Select tool for color analysis",
    n_results=3
)
context = "\n\n".join([r['content'] for r in tool_context])
with_rag = chain.invoke({"context": context, "question": test_question})

print("=" * 60)
print("WITHOUT RAG:")
print("=" * 60)
print(without_rag)
print("\n" + "=" * 60)
print("WITH RAG:")
print("=" * 60)
print(with_rag)

## 9. Interactive Query

In [ ]:
def ask_with_rag(question: str, kb_type: str = "tool", n_results: int = 2):
    """Ask a question with RAG context."""
    
    # Retrieve context based on type
    if kb_type == "tool":
        results = rag.retrieve_tool_guidance(question, "Answer question", n_results=n_results)
        context = "\n\n".join([r['content'] for r in results])
    elif kb_type == "domain":
        results = rag.retrieve_domain_knowledge(question, n_results=n_results)
        context = "\n\n".join([r['content'] for r in results])
    elif kb_type == "explanation":
        results = rag.retrieve_explanation_patterns("planning", question, {}, n_results=n_results)
        context = "\n\n".join([r['content'] for r in results])
    else:
        context = "No specific context available."
    
    # Get answer
    answer = chain.invoke({"context": context, "question": question})
    
    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    print(f"Context sources: {len(results) if 'results' in locals() else 0}")
    return answer

# Try it
ask_with_rag("When should I use the python_repl tool?", kb_type="tool")

In [ ]:
# Your custom queries here
ask_with_rag("How do I explain tool selection to users?", kb_type="explanation")